In [ ]:
import os
import psutil
import torch

ram_gb = psutil.virtual_memory().total / (1024**3)
disk = psutil.disk_usage("/")

print(f"RAM: {ram_gb:.2f} GB")
print(f"Disk total: {disk.total / (1024**3):.2f} GB")
print(f"Disk free: {disk.free / (1024**3):.2f} GB")
print(f"CUDA available: {torch.cuda.is_available()}")

RAM: 12.67 GB
Disk total: 112.64 GB
Disk free: 70.28 GB
CUDA available: True


In [ ]:
%pip install -q datasets transformers accelerate sentencepiece

In [ ]:
import json
import os
import random
from collections import Counter

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import (
    LayoutLMv3Processor,
    LayoutLMv3ForTokenClassification,
    get_linear_schedule_with_warmup,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
Device: cuda
CUDA available: True
GPU: Tesla T4


In [ ]:
dataset = load_dataset("naver-clova-ix/cord-v2")
print(dataset)
print({split: len(dataset[split]) for split in dataset.keys()})
print("Columns:", dataset["train"].column_names)

README.md:   0%|          | 0.00/27.0 [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

data/train-00000-of-00004-b4aaeceff1d90e(…): reconstructing file:   0%|          |  0.00B /  490MB            

data/train-00000-of-00004-b4aaeceff1d90e(…): downloading bytes:           |  0.00B            

data/train-00001-of-00004-7dbbe248962764(…): reconstructing file:   0%|          |  0.00B /  441MB            

...

data/test-00000-of-00001-9c204eb3f4e1179(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/800 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/100 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/100 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 800
    })
    validation: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 100
    })
    test: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 100
    })
})
{'train': 800, 'validation': 100, 'test': 100}
Columns: ['image', 'ground_truth']


In [ ]:
def parse_ground_truth(raw):
    if isinstance(raw, str):
        raw = json.loads(raw)
    return raw

def get_valid_lines(sample):
    gt = parse_ground_truth(sample["ground_truth"])
    lines = gt.get("valid_line", [])
    return lines, gt

sample = dataset["train"][0]
lines, gt = get_valid_lines(sample)

print("Image size:", sample["image"].size)
print("Top-level annotation keys:", list(gt.keys()))
print("Number of valid_line entries:", len(lines))
print("First annotation line:")
print(json.dumps(lines[0] if lines else {}, indent=2, ensure_ascii=False)[:1800])

Image size: (864, 1296)
Top-level annotation keys: ['gt_parse', 'meta', 'valid_line', 'roi', 'repeating_symbol', 'dontcare']
Number of valid_line entries: 71
First annotation line:
{
  "words": [
    {
      "quad": {
        "x2": 244,
        "y3": 390,
        "x3": 244,
        "y4": 390,
        "x1": 232,
        "y1": 372,
        "x4": 232,
        "y2": 372
      },
      "is_key": 0,
      "row_id": 2179893,
      "text": "1"
    },
    {
      "quad": {
        "x2": 270,
        "y3": 390,
        "x3": 270,
        "y4": 390,
        "x1": 256,
        "y1": 374,
        "x4": 256,
        "y2": 374
      },
      "is_key": 0,
      "row_id": 2179893,
      "text": "x"
    }
  ],
  "category": "menu.cnt",
  "group_id": 3,
  "sub_group_id": 0
}


In [ ]:
category_counts = Counter()

for row in dataset["train"]:
    valid_lines, _ = get_valid_lines(row)
    for line in valid_lines:
        category = line.get("category")
        if category:
            category_counts[category] += 1

categories = sorted(category_counts)
label_list = ["O"]
for category in categories:
    label_list.extend([f"B-{category}", f"I-{category}"])

label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

print("Semantic categories:", len(categories))
print("BIO labels including O:", len(label_list))
print("Most common categories:")
for name, count in category_counts.most_common(15):
    print(f"{name}: {count}")
print("\nExample labels:", label_list[:12])

Semantic categories: 29
BIO labels including O: 59
Most common categories:
menu.nm: 2100
menu.price: 2093
menu.cnt: 1903
total.total_price: 784
menu.unitprice: 621
sub_total.subtotal_price: 541
total.cashprice: 526
total.changeprice: 502
sub_total.tax_price: 362
menu.sub.nm: 319
total.menuqty_cnt: 233
menu.sub.cnt: 142
total.creditcardprice: 124
menu.sub.price: 123
sub_total.etc: 112

Example labels: ['O', 'B-menu.cnt', 'I-menu.cnt', 'B-menu.discountprice', 'I-menu.discountprice', 'B-menu.etc', 'I-menu.etc', 'B-menu.itemsubtotal', 'I-menu.itemsubtotal', 'B-menu.nm', 'I-menu.nm', 'B-menu.num']


In [ ]:
def normalize_box_from_quad(quad, image_width, image_height):
    xs = [quad.get(k, 0) for k in ("x1", "x2", "x3", "x4")]
    ys = [quad.get(k, 0) for k in ("y1", "y2", "y3", "y4")]

    left = max(0, min(1000, int(1000 * min(xs) / image_width)))
    top = max(0, min(1000, int(1000 * min(ys) / image_height)))
    right = max(0, min(1000, int(1000 * max(xs) / image_width)))
    bottom = max(0, min(1000, int(1000 * max(ys) / image_height)))

    # Keep boxes valid even if annotation coordinates are degenerate.
    right = max(left, right)
    bottom = max(top, bottom)
    return [left, top, right, bottom]

def extract_words_boxes_wordlabels(sample):
    image = sample["image"].convert("RGB")
    image_width, image_height = image.size
    valid_lines, _ = get_valid_lines(sample)

    words, boxes, word_label_ids = [], [], []

    for line in valid_lines:
        category = line.get("category")
        line_words = line.get("words", [])
        if not category:
            continue

        for word_index, word_obj in enumerate(line_words):
            text = str(word_obj.get("text", "")).strip()
            if not text:
                continue

            quad = word_obj.get("quad", {})
            box = normalize_box_from_quad(quad, image_width, image_height)

            prefix = "B" if word_index == 0 else "I"
            label = f"{prefix}-{category}"
            if label not in label2id:
                # A category unseen in train should not silently create a new label.
                label = "O"

            words.append(text)
            boxes.append(box)
            word_label_ids.append(label2id[label])

    return image, words, boxes, word_label_ids

img, words, boxes, word_label_ids = extract_words_boxes_wordlabels(dataset["train"][0])
print("Words:", len(words))
print("First 25 words:", words[:25])
print("First 5 boxes:", boxes[:5])
print("First 25 word labels:", [id2label[x] for x in word_label_ids[:25]])
assert len(words) == len(boxes) == len(word_label_ids)

Words: 135
First 25 words: ['1', 'x', 'Nasi', 'Campur', 'Bali', '75,000', '1', 'x', 'Bbk', 'Bengil', 'Nasi', '125,000', '1', 'x', 'MilkShake', 'Starwb', '37,000', '1', 'x', 'Ice', 'Lemon', 'Tea', '24,000', '1', 'x']
First 5 boxes: [[268, 287, 282, 300], [296, 288, 312, 300], [349, 283, 409, 300], [420, 281, 509, 301], [516, 279, 577, 299]]
First 25 word labels: ['B-menu.cnt', 'I-menu.cnt', 'B-menu.nm', 'I-menu.nm', 'I-menu.nm', 'B-menu.price', 'B-menu.cnt', 'I-menu.cnt', 'B-menu.nm', 'I-menu.nm', 'I-menu.nm', 'B-menu.price', 'B-menu.cnt', 'I-menu.cnt', 'B-menu.nm', 'I-menu.nm', 'B-menu.price', 'B-menu.cnt', 'I-menu.cnt', 'B-menu.nm', 'I-menu.nm', 'I-menu.nm', 'B-menu.price', 'B-menu.cnt', 'I-menu.cnt']


In [ ]:
MODEL_NAME = "microsoft/layoutlmv3-base"

processor = LayoutLMv3Processor.from_pretrained(
    MODEL_NAME,
    apply_ocr=False
)

print("Processor loaded:", MODEL_NAME)

preprocessor_config.json:   0%|          | 0.00/275 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/856 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.14k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

Processor loaded: microsoft/layoutlmv3-base


In [ ]:
encoded = processor(
    images=img,
    text=words,
    boxes=boxes,
    word_labels=word_label_ids,
    truncation=True,
    padding="max_length",
    max_length=512,
    return_tensors="pt",
)

print({key: tuple(value.shape) for key, value in encoded.items()})
print("Input IDs:", encoded["input_ids"].shape)
print("Labels:", encoded["labels"].shape)
print("Non-ignored token labels:", int((encoded["labels"] != -100).sum()))

{'input_ids': (1, 512), 'attention_mask': (1, 512), 'bbox': (1, 512, 4), 'labels': (1, 512), 'pixel_values': (1, 3, 224, 224)}
Input IDs: torch.Size([1, 512])
Labels: torch.Size([1, 512])
Non-ignored token labels: 135


In [ ]:
class CORDLayoutLMv3Dataset(Dataset):
    def __init__(self, split, processor, max_length=512):
        self.split = split
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.split)

    def __getitem__(self, index):
        sample = self.split[index]
        image, words, boxes, word_label_ids = extract_words_boxes_wordlabels(sample)

        encoded = self.processor(
            images=image,
            text=words,
            boxes=boxes,
            word_labels=word_label_ids,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        return {key: value.squeeze(0) for key, value in encoded.items()}

train_dataset = CORDLayoutLMv3Dataset(dataset["train"], processor)
val_dataset = CORDLayoutLMv3Dataset(dataset["validation"], processor)
test_dataset = CORDLayoutLMv3Dataset(dataset["test"], processor)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 800
Validation: 100
Test: 100


In [ ]:
item = train_dataset[0]
print({key: (tuple(value.shape), str(value.dtype)) for key, value in item.items()})
assert item["input_ids"].shape[0] == 512
assert item["bbox"].shape == (512, 4)
assert item["labels"].shape[0] == 512
assert torch.isfinite(item["pixel_values"]).all()
print("Dataset item validation passed.")

{'input_ids': ((512,), 'torch.int64'), 'attention_mask': ((512,), 'torch.int64'), 'bbox': ((512, 4), 'torch.int64'), 'labels': ((512,), 'torch.int64'), 'pixel_values': ((3, 224, 224), 'torch.float32')}
Dataset item validation passed.


<!-- if torch.cuda.is_available():
    from google.colab import drive
    drive.mount("/content/drive")

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    total_steps = NUM_EPOCHS * len(train_loader)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=max(0, int(0.1 * total_steps)),
        num_training_steps=total_steps,
    )

    history = {"train_loss": [], "val_loss": []}

    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0.0

        for step, batch in enumerate(train_loader, start=1):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad(set_to_none=True)

            outputs = model(**batch)
            loss = outputs.loss
            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite loss at epoch {epoch + 1}, step {step}")

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            running_loss += loss.item()
            if step % 25 == 0:
                print(
                    f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
                    f"Step {step}/{len(train_loader)} | "
                    f"Loss {loss.item():.4f}"
                )

        train_loss = running_loss / max(1, len(train_loader))
        history["train_loss"].append(train_loss)

        model.eval()
        val_running_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                val_running_loss += model(**batch).loss.item()

        val_loss = val_running_loss / max(1, len(val_loader))
        history["val_loss"].append(val_loss)

        print(
            f"Epoch {epoch + 1}: train_loss={train_loss:.4f}, "
            f"val_loss={val_loss:.4f}"
        )

        model.save_pretrained(OUTPUT_DIR)
        processor.save_pretrained(OUTPUT_DIR)
        with open(os.path.join(OUTPUT_DIR, "label_list.json"), "w") as f:
            json.dump(label_list, f, indent=2)

    with open(os.path.join(OUTPUT_DIR, "history.json"), "w") as f:
        json.dump(history, f, indent=2)

    print("Saved model and history to:", OUTPUT_DIR)
else:
    print("Skipped training because CUDA is unavailable.") -->

In [ ]:
model = LayoutLMv3ForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

model.to(DEVICE)
print("Model loaded.")
print("Number of labels:", model.config.num_labels)

model.safetensors: reconstructing file:   0%|          |  0.00B /  501MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

[transformers] LayoutLMv3ForTokenClassification LOAD REPORT from: microsoft/layoutlmv3-base
Key                        | Status  | 
---------------------------+---------+-
classifier.dense.weight    | MISSING | 
classifier.out_proj.bias   | MISSING | 
classifier.out_proj.weight | MISSING | 
classifier.dense.bias      | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded.
Number of labels: 59


In [ ]:
OUTPUT_DIR = "/content/drive/MyDrive/DocVision/layoutlmv3-cord"
NUM_EPOCHS = 3
BATCH_SIZE = 2 if torch.cuda.is_available() else 1
LEARNING_RATE = 2e-5
MAX_LENGTH = 512

if torch.cuda.is_available():
    print("GPU training is enabled.")
    print("Batch size:", BATCH_SIZE)
else:
    print("CUDA is unavailable. Training will not start yet.")
    print("You can complete data validation and reconnect to a GPU later.")

GPU training is enabled.
Batch size: 2


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
import torch
from transformers import (
    LayoutLMv3Processor,
    LayoutLMv3ForTokenClassification,
)

OUTPUT_DIR = "/content/drive/MyDrive/DocVision/layoutlmv3-cord"

print("Checkpoint exists:", os.path.isdir(OUTPUT_DIR))
print("Files:", os.listdir(OUTPUT_DIR))

processor = LayoutLMv3Processor.from_pretrained(OUTPUT_DIR)
model = LayoutLMv3ForTokenClassification.from_pretrained(OUTPUT_DIR)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(DEVICE)
model.eval()

print("Saved model loaded. Device:", DEVICE)

Mounted at /content/drive
Checkpoint exists: True
Files: ['history.json', 'processor_config.json', 'config.json', 'tokenizer.json', 'tokenizer_config.json', 'label_list.json', 'model.safetensors']


Loading weights:   0%|          | 0/216 [00:00<?, ?it/s]

Saved model loaded. Device: cuda


In [ ]:
import torch
from torch.utils.data import DataLoader

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

def extract_entities(labels):
    entities = set()
    entity_type = None
    start = None

    def close_entity(end):
        if entity_type is not None:
            entities.add((entity_type, start, end))

    for i, label in enumerate(labels):
        if label == "O":
            close_entity(i)
            entity_type, start = None, None
            continue

        prefix, category = label.split("-", 1)

        if prefix == "B" or entity_type != category:
            close_entity(i)
            entity_type, start = category, i

    close_entity(len(labels))
    return entities


model.eval()

true_positive = 0
false_positive = 0
false_negative = 0

with torch.no_grad():
    for batch in test_loader:
        batch = {
            k: v.to(DEVICE)
            for k, v in batch.items()
        }

        logits = model(**batch).logits
        predictions = logits.argmax(dim=-1)

        for pred_row, label_row in zip(
            predictions.cpu().tolist(),
            batch["labels"].cpu().tolist(),
        ):
            true_labels = []
            pred_labels = []

            for pred_id, true_id in zip(pred_row, label_row):
                if true_id == -100:
                    continue

                true_labels.append(id2label[true_id])
                pred_labels.append(id2label[pred_id])

            true_entities = extract_entities(true_labels)
            pred_entities = extract_entities(pred_labels)

            true_positive += len(true_entities & pred_entities)
            false_positive += len(pred_entities - true_entities)
            false_negative += len(true_entities - pred_entities)

precision = true_positive / max(
    1, true_positive + false_positive
)
recall = true_positive / max(
    1, true_positive + false_negative
)
f1 = 2 * precision * recall / max(
    1e-12, precision + recall
)

print(f"Entity precision: {precision:.4f}")
print(f"Entity recall:    {recall:.4f}")
print(f"Entity F1:        {f1:.4f}")
print("Exact entity matches:", true_positive)
print("False positives:", false_positive)
print("False negatives:", false_negative)

Entity precision: 0.8873
Entity recall:    0.9083
Entity F1:        0.8977
Exact entity matches: 1189
False positives: 151
False negatives: 120


In [ ]:
print("Dataset class:", type(test_dataset))
print("Number of test samples:", len(test_dataset))

sample = test_dataset[0]

print("\nSample keys:", sample.keys())

for key, value in sample.items():
    if hasattr(value, "shape"):
        print(f"{key}: shape={value.shape}, dtype={value.dtype}")
    elif isinstance(value, list):
        print(f"{key}: list length={len(value)}")
    else:
        print(f"{key}: type={type(value)}")

Dataset class: <class '__main__.CORDLayoutLMv3Dataset'>
Number of test samples: 100

Sample keys: dict_keys(['input_ids', 'attention_mask', 'bbox', 'labels', 'pixel_values'])
input_ids: shape=torch.Size([512]), dtype=torch.int64
attention_mask: shape=torch.Size([512]), dtype=torch.int64
bbox: shape=torch.Size([512, 4]), dtype=torch.int64
labels: shape=torch.Size([512]), dtype=torch.int64
pixel_values: shape=torch.Size([3, 224, 224]), dtype=torch.float32


In [ ]:
BASE_DIR = "/content/drive/MyDrive/DocVision/layoutlmv3-cord"
OUTPUT_DIR = "/content/drive/MyDrive/DocVision/layoutlmv3-cord-5epoch"

NUM_EPOCHS = 2  # Two additional epochs: 3 + 2 = 5
BATCH_SIZE = 2
LEARNING_RATE = 2e-5
MAX_LENGTH = 512

In [ ]:
import torch
from transformers import (
    LayoutLMv3Processor,
    LayoutLMv3ForTokenClassification,
)

processor = LayoutLMv3Processor.from_pretrained(BASE_DIR)
model = LayoutLMv3ForTokenClassification.from_pretrained(BASE_DIR)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(DEVICE)

print("Loaded existing checkpoint:", BASE_DIR)
print("Device:", DEVICE)

Loading weights:   0%|          | 0/216 [00:00<?, ?it/s]

Loaded existing checkpoint: /content/drive/MyDrive/DocVision/layoutlmv3-cord
Device: cuda


In [ ]:

import os
import json
import torch
from torch.utils.data import DataLoader
from transformers import get_linear_schedule_with_warmup

# Existing 3-epoch checkpoint
BASE_DIR = "/content/drive/MyDrive/DocVision/layoutlmv3-cord"

# Save the updated model separately
OUTPUT_DIR = "/content/drive/MyDrive/DocVision/layoutlmv3-cord-5epoch"

# Two additional epochs: 3 + 2 = 5
ADDITIONAL_EPOCHS = 2
BATCH_SIZE = 2
LEARNING_RATE = 2e-5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime in Colab before training.")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load the original trained checkpoint if not already loaded
from transformers import (
    LayoutLMv3Processor,
    LayoutLMv3ForTokenClassification,
)

processor = LayoutLMv3Processor.from_pretrained(BASE_DIR)
model = LayoutLMv3ForTokenClassification.from_pretrained(BASE_DIR)
model.to(DEVICE)

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

# Optimizer and scheduler for the additional training
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
)

total_steps = ADDITIONAL_EPOCHS * len(train_loader)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, int(0.1 * total_steps)),
    num_training_steps=total_steps,
)

history = {"train_loss": [], "val_loss": []}
best_val_loss = float("inf")

for epoch in range(ADDITIONAL_EPOCHS):
    model.train()
    running_loss = 0.0

    for step, batch in enumerate(train_loader, start=1):
        batch = {
            key: value.to(DEVICE)
            for key, value in batch.items()
        }

        optimizer.zero_grad(set_to_none=True)
        outputs = model(**batch)
        loss = outputs.loss

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite loss at additional epoch {epoch + 1}, "
                f"step {step}"
            )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()

        if step % 25 == 0:
            print(
                f"Additional epoch {epoch + 1}/{ADDITIONAL_EPOCHS} | "
                f"Step {step}/{len(train_loader)} | "
                f"Loss {loss.item():.4f}"
            )

    train_loss = running_loss / max(1, len(train_loader))
    history["train_loss"].append(train_loss)

    # Validation
    model.eval()
    val_running_loss = 0.0

    with torch.no_grad():
        for batch in val_loader:
            batch = {
                key: value.to(DEVICE)
                for key, value in batch.items()
            }
            val_running_loss += model(**batch).loss.item()

    val_loss = val_running_loss / max(1, len(val_loader))
    history["val_loss"].append(val_loss)

    # Save the latest checkpoint after each epoch
    model.save_pretrained(OUTPUT_DIR)
    processor.save_pretrained(OUTPUT_DIR)

    with open(os.path.join(OUTPUT_DIR, "label_list.json"), "w") as f:
        json.dump(label_list, f, indent=2)

    with open(os.path.join(OUTPUT_DIR, "history.json"), "w") as f:
        json.dump(history, f, indent=2)

    print(
        f"Additional epoch {epoch + 1}: "
        f"train_loss={train_loss:.4f}, "
        f"val_loss={val_loss:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss

print("Additional fine-tuning complete.")
print("Checkpoint saved to:", OUTPUT_DIR)
print("Best validation loss during these two epochs:", best_val_loss)


Loading weights:   0%|          | 0/216 [00:00<?, ?it/s]

Additional epoch 1/2 | Step 25/400 | Loss 0.0289
Additional epoch 1/2 | Step 50/400 | Loss 0.7919
Additional epoch 1/2 | Step 75/400 | Loss 0.0638
Additional epoch 1/2 | Step 100/400 | Loss 0.9051
Additional epoch 1/2 | Step 125/400 | Loss 0.5600
Additional epoch 1/2 | Step 150/400 | Loss 0.0388
Additional epoch 1/2 | Step 175/400 | Loss 0.4723
Additional epoch 1/2 | Step 200/400 | Loss 0.1208
Additional epoch 1/2 | Step 225/400 | Loss 0.2271
Additional epoch 1/2 | Step 250/400 | Loss 0.7000
Additional epoch 1/2 | Step 275/400 | Loss 0.1909
Additional epoch 1/2 | Step 300/400 | Loss 0.3382
Additional epoch 1/2 | Step 325/400 | Loss 0.0846
Additional epoch 1/2 | Step 350/400 | Loss 0.2449
Additional epoch 1/2 | Step 375/400 | Loss 0.5951
Additional epoch 1/2 | Step 400/400 | Loss 0.1100


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Additional epoch 1: train_loss=0.3719, val_loss=0.2807
Additional epoch 2/2 | Step 25/400 | Loss 1.1125
Additional epoch 2/2 | Step 50/400 | Loss 0.0932
Additional epoch 2/2 | Step 75/400 | Loss 0.0641
Additional epoch 2/2 | Step 100/400 | Loss 0.2906
Additional epoch 2/2 | Step 125/400 | Loss 0.8207
Additional epoch 2/2 | Step 150/400 | Loss 0.4432
Additional epoch 2/2 | Step 175/400 | Loss 0.0166
Additional epoch 2/2 | Step 200/400 | Loss 0.1834
Additional epoch 2/2 | Step 225/400 | Loss 0.3256
Additional epoch 2/2 | Step 250/400 | Loss 0.0460
Additional epoch 2/2 | Step 275/400 | Loss 0.2314
Additional epoch 2/2 | Step 300/400 | Loss 0.3751
Additional epoch 2/2 | Step 325/400 | Loss 0.1012
Additional epoch 2/2 | Step 350/400 | Loss 0.5295
Additional epoch 2/2 | Step 375/400 | Loss 0.3129
Additional epoch 2/2 | Step 400/400 | Loss 0.0164


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Additional epoch 2: train_loss=0.2325, val_loss=0.2321
Additional fine-tuning complete.
Checkpoint saved to: /content/drive/MyDrive/DocVision/layoutlmv3-cord-5epoch
Best validation loss during these two epochs: 0.2320516016613692


In [ ]:
NEW_DIR = "/content/drive/MyDrive/DocVision/layoutlmv3-cord-5epoch"

processor = LayoutLMv3Processor.from_pretrained(NEW_DIR)
model = LayoutLMv3ForTokenClassification.from_pretrained(NEW_DIR)

model.to(DEVICE)
model.eval()

print("Five-epoch model loaded.")

Loading weights:   0%|          | 0/216 [00:00<?, ?it/s]

Five-epoch model loaded.


In [ ]:
import torch
from torch.utils.data import DataLoader

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

def extract_entities(labels):
    entities = set()
    entity_type = None
    start = None

    def close_entity(end):
        if entity_type is not None:
            entities.add((entity_type, start, end))

    for i, label in enumerate(labels):
        if label == "O":
            close_entity(i)
            entity_type, start = None, None
            continue

        prefix, category = label.split("-", 1)

        if prefix == "B" or entity_type != category:
            close_entity(i)
            entity_type, start = category, i

    close_entity(len(labels))
    return entities


model.eval()

true_positive = 0
false_positive = 0
false_negative = 0

with torch.no_grad():
    for batch in test_loader:
        batch = {
            k: v.to(DEVICE)
            for k, v in batch.items()
        }

        logits = model(**batch).logits
        predictions = logits.argmax(dim=-1)

        for pred_row, label_row in zip(
            predictions.cpu().tolist(),
            batch["labels"].cpu().tolist(),
        ):
            true_labels = []
            pred_labels = []

            for pred_id, true_id in zip(pred_row, label_row):
                if true_id == -100:
                    continue

                true_labels.append(id2label[true_id])
                pred_labels.append(id2label[pred_id])

            true_entities = extract_entities(true_labels)
            pred_entities = extract_entities(pred_labels)

            true_positive += len(true_entities & pred_entities)
            false_positive += len(pred_entities - true_entities)
            false_negative += len(true_entities - pred_entities)

precision = true_positive / max(
    1, true_positive + false_positive
)
recall = true_positive / max(
    1, true_positive + false_negative
)
f1 = 2 * precision * recall / max(
    1e-12, precision + recall
)

print(f"Entity precision: {precision:.4f}")
print(f"Entity recall:    {recall:.4f}")
print(f"Entity F1:        {f1:.4f}")
print("Exact entity matches:", true_positive)
print("False positives:", false_positive)
print("False negatives:", false_negative)

Entity precision: 0.9258
Entity recall:    0.9442
Entity F1:        0.9349
Exact entity matches: 1236
False positives: 99
False negatives: 73
